# Hands-on Projects — Reranking

**Module:** 03 — Reranking

Build a retrieve-and-rerank demo with an eval harness, then harden it toward something you could show in a portfolio or team review.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement a mini hybrid retrieve + toy/real CE rerank
- Create a graded mini eval set and compute metrics
- Add logging, fallback, and citation packing
- Complete at least one extension track (cache, API, cascade)
- Present before/after metrics cleanly


## Project: Retrieve & Rerank Demo

**Definition.** A **vertical slice** that retrieves from a tiny corpus, reranks, packs citations, and reports offline metrics—plus optional LLM answer generation.

**Why it matters.** End-to-end learning beats isolated toy cells; you practice the interfaces you'll use at work.

**How it works.** Corpus → retrieve → rerank → pack → (optional LLM) → eval harness → writeup of lifts.

**Intuition.** Ship a thin wedge: one path that works completely.

**Common pitfalls.**
- Polishing UI before metrics exist
- Hard-coding secrets in the notebook
- No before/after comparison

**When to use.** Capstone for this module; reuse patterns in the RAG module.

### Project ideas

1. Support FAQ retrieve-and-rerank with nDCG poster
2. Multilingual slice bakeoff (EN vs one other language)
3. Cascade: lexical filter → MiniLM → optional API rerank
4. Citation UI mock: show packed chunks beside the answer

### Deliverable checklist

- [ ] Corpus + retrieve
- [ ] Rerank stage with before/after lists
- [ ] Offline metric table
- [ ] Latency note for chosen N
- [ ] Fallback behavior documented in comments
- [ ] No secrets committed


In [ ]:
# Demo 1 — tiny corpus + lexical retrieve
CORPUS = {
    "p1": "Refunds are available within 60 days of purchase with receipt.",
    "p2": "Clearance items are final sale and not refundable.",
    "p3": "Standard shipping takes 3 to 5 business days.",
    "p4": "Reset your password using the email link under Account settings.",
    "p5": "Students get a 10% refund credit on annual plans.",
}

def retrieve(q, k=4):
    qw = set(q.lower().split())
    scored = []
    for i, t in CORPUS.items():
        s = len(qw & set(t.lower().split()))
        scored.append((s, i, t))
    scored.sort(reverse=True)
    return scored[:k]

for row in retrieve("What is the refund window?"):
    print(row)


In [ ]:
# Demo 2 — toy cross-encoder + pack
def ce_score(q, doc):
    qset, dset = set(q.lower().split()), set(doc.lower().split())
    overlap = len(qset & dset) / (len(qset) + 1e-9)
    bonus = 0.3 if "60" in doc or "refund" in doc.lower() else 0.0
    return overlap + bonus

def rerank(q, hits):
    return sorted(hits, key=lambda r: ce_score(q, r[2]), reverse=True)

q = "What is the refund window?"
hits = retrieve(q, k=5)
reranked = rerank(q, hits)
print("BEFORE", [h[1] for h in hits])
print("AFTER ", [h[1] for h in reranked])
for s, i, t in reranked[:2]:
    print(f"[{i}] {t}")


In [ ]:
# Demo 3 — mini eval harness
EVAL = [
    {"q": "refund window", "gold": ["p1"]},
    {"q": "clearance return", "gold": ["p2"]},
    {"q": "shipping days", "gold": ["p3"]},
]

def recall_at_k(ranked_ids, gold, k=2):
    return len(set(ranked_ids[:k]) & set(gold)) / len(gold)

def run(use_rerank=True):
    scores = []
    for row in EVAL:
        hits = retrieve(row["q"], k=4)
        if use_rerank:
            hits = rerank(row["q"], hits)
        ids = [h[1] for h in hits]
        scores.append(recall_at_k(ids, row["gold"], k=2))
        print(row["q"], ids, scores[-1])
    print("macro", sum(scores)/len(scores))

print("== retrieve only ==")
run(False)
print("== + rerank ==")
run(True)


In [ ]:
# Demo 4 — optional hosted rerank JSON (no network)
import json
YOUR_API_KEY = "YOUR_API_KEY"
payload = {
  "model": "rerank-english-v3.0",
  "query": "refund window",
  "documents": list(CORPUS.values()),
  "top_n": 3,
}
print(json.dumps(payload, indent=2)[:400])
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


### Try it yourself — Project: Retrieve & Rerank Demo

1. Track A: Grow CORPUS to ≥15 passages and EVAL to ≥10 queries.
2. Track B: Swap toy CE for SentenceTransformers MiniLM locally.
3. Track C: Add RRF hybrid (BM25-like + embedding toy vectors).
4. Track D: Add score cache + timeout fallback.


## Portfolio hardening

**Definition.** Turn the demo into a reviewable artifact: seeds, metrics, traces, and a short narrative of what improved.

**Why it matters.** Interviewers and teammates trust measured deltas over screenshots alone.

**How it works.** Freeze EVAL; log traces; plot one chart; list limitations honestly.

**Intuition.** Show the scientific method, not only the happy path.

**Common pitfalls.**
- Only cherry-picked queries in the writeup
- Hidden manual tuning on the test set

**When to use.** When sharing externally or handing off to a team.

### Further reading directions

- MS MARCO passage ranking papers / leaderboards
- BEIR / MTEB retrieval evaluation suites
- Vendor docs: Cohere Rerank, Jina rerankers, BGE model cards
- Module 04 RAG notebooks for full pipeline context


In [ ]:
# Demo 1 — trace export
import json
trace = {
  "query": "refund window",
  "retrieved": ["p5", "p1", "p2"],
  "reranked": ["p1", "p2", "p5"],
  "packed": ["p1", "p2"],
  "latency_ms": {"retrieve": 12, "rerank": 45},
}
print(json.dumps(trace, indent=2))


In [ ]:
# Demo 2 — results table
rows = [
    ("retrieve", 0.55),
    ("+toy CE", 0.80),
    ("+hybrid", 0.85),
]
print("system | Recall@2")
for n, s in rows:
    print(f"{n:10s}| {s:.2f}")


In [ ]:
# Demo 3 — secret hygiene
import os
key = os.getenv("COHERE_API_KEY", "YOUR_API_KEY")
print("using", key[:8] + "...")


### Try it yourself — Portfolio hardening

1. Write a 5-bullet limitations section for your demo.
2. Record three failed queries and classify the stage.


## Glossary

- **vertical slice**: Thin end-to-end implementation
- **eval harness**: Fixed queries + metrics
- **hardening**: Logs, fallbacks, secret hygiene


### Workshop drill — Hands-on Projects — Reranking (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Hands-on Projects — Reranking
headings = ['Project: Retrieve & Rerank Demo', 'Portfolio hardening']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Hands-on Projects — Reranking (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Hands-on Projects — Reranking
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Hands-on Projects — Reranking (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Hands-on Projects — Reranking
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Hands-on Projects — Reranking (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Hands-on Projects — Reranking
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


### Workshop drill — Hands-on Projects — Reranking (5)

Compare bi-encoder vs cross-encoder in a small table (fill TODOs).


In [ ]:
# Workshop drill 5 — Hands-on Projects — Reranking
print('| axis | bi | cross |')
print('|------|----|-------|')
print('| latency | TODO | TODO |')
print('| precision | TODO | TODO |')


### Workshop drill — Hands-on Projects — Reranking (6)

Simulate score calibration: min-max normalize three reranker scores.


In [ ]:
# Workshop drill 6 — Hands-on Projects — Reranking
scores = [2.1, -0.4, 0.8]
lo, hi = min(scores), max(scores)
norm = [(s-lo)/(hi-lo+1e-9) for s in scores]
print(norm)


## Summary & Key Takeaways

- A vertical retrieve→rerank→eval slice beats disconnected snippets
- Always report before/after metrics and latency
- Extension tracks: real CE, hybrid, cache, API
- Hardening means traces, gates, and honest failure cases

### Practice

Finish Track A and one of B/C/D; present a metric table.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
